# Персональные PDF-отчёты для преподавателей

Генерирует один PDF на каждого преподавателя.

**Каждый PDF содержит:**
- **Раздел А** — общие метрики по программам преподавателя из общего опроса:
  - Удовлетворённость образовательным процессом, преподавателями, совпадение ожиданий
  - Критерии оценивания, порядок сдачи, соответствие оценивания критериям
  - Поддержка и понятность преподавательской команды
- **Раздел Б** — персональные оценки преподавателя: среднее, медиана, распределение, программы

**Данные других преподавателей нигде не фигурируют.**

**Выходные файлы:** `output/teacher_pdfs/<Имя>.pdf`

**Зависимости:** `pip install pandas numpy matplotlib`

In [ ]:
import os
import re
import warnings
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from matplotlib.backends.backend_pdf import PdfPages

warnings.filterwarnings("ignore")
matplotlib.rcParams["pdf.fonttype"] = 42   # embed fonts (no Type 3)
print("Библиотеки загружены.")

In [ ]:
# ── Пути к файлам ──────────────────────────────────────────────────────────
# Запускайте из папки проекта (рядом с combined_general_agg.csv),
# или скорректируйте пути ниже.
BASE_DIR     = Path("..")                                        # если notebook в папке notebooks/
GENERAL_CSV  = BASE_DIR / "combined_general_agg.csv"
TEACHERS_CSV = BASE_DIR / "data/processed/combined_teachers_agg.csv"
OUTPUT_DIR   = BASE_DIR / "output/teacher_pdfs"

# ── Параметры фильтрации ───────────────────────────────────────────────────
MIN_RESPONSES = 3        # преподаватели с менее чем N оценками пропускаются
ONLY_TEACHER  = None     # None = все, или str: "Иванов Иван" = только один

print(f"Данные:     {GENERAL_CSV}")
print(f"Учителя:    {TEACHERS_CSV}")
print(f"PDF-папка:  {OUTPUT_DIR}")

In [ ]:
# ── Цветовая палитра ───────────────────────────────────────────────────────
PRIMARY   = "#2B4590"
ACCENT    = "#E8572A"
GREEN     = "#3A9E72"
AMBER     = "#F5C518"
LIGHT_BG  = "#F5F7FA"
GRAY      = "#9CA3AF"
TEXT_DARK = "#1F2937"
TEXT_MID  = "#4B5563"

plt.rcParams.update({
    "font.family":        "sans-serif",
    "axes.spines.top":    False,
    "axes.spines.right":  False,
    "axes.facecolor":     "white",
    "figure.facecolor":   "white",
    "text.color":         TEXT_DARK,
    "axes.labelcolor":    TEXT_MID,
    "xtick.color":        TEXT_MID,
    "ytick.color":        TEXT_MID,
})

# ── Метрики раздела А и их параметры ──────────────────────────────────────
# (название_колонки, заголовок карточки, max_шкалы)
SECTION_A_METRICS = [
    # — Общая удовлетворённость образовательным процессом —
    ("satisf_overall",          "Удовлетворённость\nобразовательным процессом",  4),
    ("satisf_teachers",         "Удовлетворённость\nпреподавателями",            4),
    ("expect_match",            "Совпадение ожиданий\nс опытом",                 4),
    # — Оценивание —
    ("assess_criteria_timely",  "Критерии оценивания\nпредставлены вовремя",     5),
    ("assess_order_clear",      "Порядок сдачи\nи оценивания понятен",           5),
    ("assess_consistent",       "Оценивание соответствует\nкритериям",           5),
    # — Преподавательская команда —
    ("fac_support_score",       "Поддержка преподавательской\nкоманды",          4),
    ("fac_clarity_score",       "Понятность объяснений\nпреподавателей",         4),
]

COLS_PER_ROW_A = 3   # количество карточек в строке в разделе А

print(f"{len(SECTION_A_METRICS)} метрик в разделе А, по {COLS_PER_ROW_A} в строке.")

In [ ]:
# ── Вспомогательные функции ────────────────────────────────────────────────

def safe_filename(name: str) -> str:
    """Имя файла без спецсимволов."""
    return re.sub(r'[^\w\s\-]', '', name).strip().replace(' ', '_')


def fmt_score(v, scale: int) -> str:
    if pd.isna(v):
        return "—"
    return f"{v:.2f} / {scale}"


def score_color(v: float, scale: int) -> str:
    """Цвет по доле от шкалы: ≥85% → зелёный, ≥65% → жёлтый, иначе красный."""
    if pd.isna(v):
        return GRAY
    pct = v / scale
    if pct >= 0.85:
        return GREEN
    elif pct >= 0.65:
        return AMBER
    return ACCENT


def scale_label(v: float, scale: int) -> str:
    if pd.isna(v):
        return "нет данных"
    pct = v / scale
    if pct >= 0.85:
        return "Отлично"
    elif pct >= 0.70:
        return "Хорошо"
    elif pct >= 0.55:
        return "Удовл."
    return "Требует внимания"


def report_date_str() -> str:
    months = {
        1: "января", 2: "февраля", 3: "марта", 4: "апреля",
        5: "мая", 6: "июня", 7: "июля", 8: "августа",
        9: "сентября", 10: "октября", 11: "ноября", 12: "декабря",
    }
    d = datetime.now()
    return f"{d.day} {months[d.month]} {d.year} г."


print("Вспомогательные функции определены.")

In [ ]:
# ── Функции отрисовки элементов PDF ───────────────────────────────────────

def draw_header(ax, teacher_name: str, report_date: str, programs: list):
    ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis("off")
    ax.add_patch(mpatches.FancyBboxPatch(
        (0, 0), 1, 1, boxstyle="round,pad=0.01",
        facecolor=PRIMARY, edgecolor="none",
        transform=ax.transAxes, clip_on=False,
    ))
    ax.text(0.04, 0.75, "Персональный отчёт · SFQ 2025-26",
            color="white", fontsize=9, alpha=0.80)
    ax.text(0.04, 0.30, teacher_name,
            color="white", fontsize=17, fontweight="bold")
    prog_str = " · ".join(programs[:4])
    if len(programs) > 4:
        prog_str += f" и ещё {len(programs) - 4}"
    ax.text(0.96, 0.75, prog_str, color="white", fontsize=8, alpha=0.80, ha="right")
    ax.text(0.96, 0.30, report_date,  color="white", fontsize=9, alpha=0.70, ha="right")


def draw_section_title(ax, title: str, letter: str = ""):
    ax.axis("off"); ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    if letter:
        ax.add_patch(mpatches.FancyBboxPatch(
            (0, 0.05), 0.028, 0.90,
            boxstyle="round,pad=0.01",
            facecolor=PRIMARY, edgecolor="none",
        ))
        ax.text(0.014, 0.50, letter,
                color="white", fontsize=10, ha="center", va="center", fontweight="bold")
        ax.text(0.040, 0.50, title,
                color=TEXT_DARK, fontsize=10.5, va="center", fontweight="bold")
    else:
        ax.text(0.0, 0.50, title, color=TEXT_MID, fontsize=9, va="center")


def draw_metric_card(ax, title: str, value: str, subtitle: str = "", color: str = PRIMARY):
    ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis("off")
    ax.add_patch(mpatches.FancyBboxPatch(
        (0.04, 0.06), 0.92, 0.88,
        boxstyle="round,pad=0.02",
        facecolor=LIGHT_BG, edgecolor=color, linewidth=1.4,
    ))
    ax.text(0.50, 0.82, title,
            ha="center", va="center", fontsize=7, color=TEXT_MID,
            wrap=True, multialignment="center")
    ax.text(0.50, 0.50, value,
            ha="center", va="center", fontsize=17, color=color, fontweight="bold")
    if subtitle:
        ax.text(0.50, 0.20, subtitle,
                ha="center", va="center", fontsize=7, color=TEXT_MID)


def draw_rating_histogram(ax, ratings: pd.Series, t_mean: float):
    counts, _ = np.histogram(ratings.dropna(), bins=range(0, 12))
    bar_colors = [PRIMARY if i >= round(t_mean) - 1 else LIGHT_BG for i in range(11)]
    ax.bar(range(0, 11), counts, color=bar_colors, edgecolor="white", linewidth=0.8)
    ax.set_xticks(range(0, 11))
    ax.set_xlabel("Оценка (0–10)", fontsize=8)
    ax.set_ylabel("Кол-во ответов", fontsize=8)
    ax.set_title("Распределение оценок студентами", fontsize=9, fontweight="bold", pad=6)
    ax.tick_params(labelsize=7)
    ax.set_xlim(-0.6, 10.6)
    ax.axvline(t_mean, color=ACCENT, linewidth=2, linestyle="--",
               label=f"Среднее: {t_mean:.2f}")
    ax.legend(fontsize=7, frameon=False)


def draw_programs_table(ax, rows: list):
    ax.axis("off")
    if not rows:
        ax.text(0.5, 0.5, "Нет данных", ha="center", va="center", color=GRAY)
        return
    headers = ["Программа", "Курс", "Ответов", "Ср. балл"]
    col_w   = [0.52, 0.14, 0.14, 0.20]
    y_top   = 0.93
    row_h   = min(0.12, 0.88 / (len(rows) + 1))

    x = 0.0
    for h, w in zip(headers, col_w):
        ax.text(x + w / 2, y_top, h, ha="center", va="center",
                fontsize=7.5, fontweight="bold", color=TEXT_DARK)
        x += w
    ax.axhline(y_top - 0.04, color=GRAY, linewidth=0.5)

    for i, row in enumerate(rows):
        y = y_top - (i + 1) * row_h - 0.02
        if y < 0.02:
            break
        bg = LIGHT_BG if i % 2 == 0 else "white"
        ax.add_patch(matplotlib.patches.Rectangle(
            (0, y - row_h * 0.45), 1, row_h * 0.90,
            facecolor=bg, edgecolor="none", zorder=0,
        ))
        vals = [
            str(row.get("program", ""))[:44],
            str(row.get("year", "")),
            str(row.get("n", "")),
            f"{row['mean']:.1f}" if row.get("mean") is not None else "—",
        ]
        x = 0.0
        for val, w in zip(vals, col_w):
            ax.text(x + w / 2, y, val, ha="center", va="center",
                    fontsize=6.5, color=TEXT_DARK)
            x += w


print("Функции отрисовки определены.")

In [ ]:
# ── Основная функция генерации PDF ────────────────────────────────────────

def generate_pdf_for_teacher(
    teacher_name: str,
    teacher_df: pd.DataFrame,
    general_df: pd.DataFrame,
    output_path: str,
) -> None:

    # ── Статистика преподавателя ───────────────────────────────────────────
    ratings  = teacher_df["rating"].dropna()
    t_mean   = ratings.mean()
    t_n      = len(ratings)
    t_median = ratings.median() if t_n > 0 else np.nan
    perc_high = (ratings >= 9).sum() / t_n * 100 if t_n else 0
    perc_low  = (ratings <= 4).sum() / t_n * 100 if t_n else 0

    program_rows = [
        {"program": prog, "year": year,
         "n": len(grp), "mean": grp["rating"].mean()}
        for (prog, year), grp in teacher_df.groupby(["program", "year"])
    ]
    program_rows.sort(key=lambda r: r["n"], reverse=True)

    # ── Контекст из общего опроса (только программы этого преподавателя) ──
    t_programs = teacher_df["program"].unique()
    t_years    = teacher_df["year"].unique()
    ctx = general_df[
        general_df["program"].isin(t_programs) &
        general_df["year"].isin(t_years)
    ]

    def ctx_mean(col):
        return ctx[col].mean() if col in ctx.columns and not ctx[col].dropna().empty else np.nan

    # Вычисляем значения всех метрик раздела А
    section_a_values = [
        (title, ctx_mean(col), scale)
        for col, title, scale in SECTION_A_METRICS
    ]

    # ── Разбивка метрик на строки ──────────────────────────────────────────
    n_rows_a = -(-len(section_a_values) // COLS_PER_ROW_A)   # ceil division
    date_str = report_date_str()
    progs_display = [r["program"] for r in program_rows]

    # ── Построение GridSpec ────────────────────────────────────────────────
    # 12 колонок для гибкого совмещения групп по 3 (раздел А) и 4 (раздел Б)
    N_GCOLS = 12

    # Строки GridSpec:
    #  0          = шапка
    #  1          = разделитель
    #  2          = заголовок раздела А
    #  3..2+nrowsA = строки карточек А
    #  3+nrowsA   = разделитель
    #  4+nrowsA   = заголовок раздела Б
    #  5+nrowsA   = строка карточек Б
    #  6+nrowsA   = разделитель
    #  7+nrowsA   = заголовок таблицы программ
    #  8+nrowsA   = таблица программ

    GR_HDR   = 0
    GR_SP1   = 1
    GR_A_TTL = 2
    GR_A_0   = 3                          # первая строка карточек А
    GR_SP2   = GR_A_0 + n_rows_a
    GR_B_TTL = GR_SP2 + 1
    GR_B_0   = GR_B_TTL + 1
    GR_SP3   = GR_B_0 + 1
    GR_TBL_T = GR_SP3 + 1
    GR_TBL   = GR_TBL_T + 1
    N_GROWS  = GR_TBL + 1

    hr = [0.0] * N_GROWS
    hr[GR_HDR]   = 1.0
    hr[GR_SP1]   = 0.15
    hr[GR_A_TTL] = 0.40
    for i in range(n_rows_a):
        hr[GR_A_0 + i] = 0.80
    hr[GR_SP2]   = 0.15
    hr[GR_B_TTL] = 0.40
    hr[GR_B_0]   = 0.80
    hr[GR_SP3]   = 0.12
    hr[GR_TBL_T] = 0.30
    hr[GR_TBL]   = 1.30

    with PdfPages(output_path) as pdf:

        # ================================================================
        # СТРАНИЦА 1: шапка + разделы А и Б + таблица программ
        # ================================================================
        fig = plt.figure(figsize=(8.27, 11.69))
        fig.subplots_adjust(left=0.05, right=0.95, top=0.97, bottom=0.03,
                            hspace=0.45, wspace=0.30)
        gs = gridspec.GridSpec(N_GROWS, N_GCOLS, figure=fig,
                               height_ratios=hr)

        # Шапка
        draw_header(fig.add_subplot(gs[GR_HDR, :]),
                    teacher_name, date_str, progs_display)

        # Заголовок раздела А
        draw_section_title(
            fig.add_subplot(gs[GR_A_TTL, :]),
            "Общие результаты опроса по программам преподавателя", "А"
        )

        # Карточки раздела А (по COLS_PER_ROW_A в строке, 12 колонок)
        card_w = N_GCOLS // COLS_PER_ROW_A   # = 4 при COLS_PER_ROW_A=3
        for idx, (title, value, scale) in enumerate(section_a_values):
            row_i = idx // COLS_PER_ROW_A
            col_i = idx % COLS_PER_ROW_A
            c_start = col_i * card_w
            c_end   = c_start + card_w
            ax = fig.add_subplot(gs[GR_A_0 + row_i, c_start:c_end])
            draw_metric_card(
                ax,
                title=title,
                value=fmt_score(value, scale),
                subtitle=scale_label(value, scale),
                color=score_color(value, scale),
            )

        # Заголовок раздела Б
        draw_section_title(
            fig.add_subplot(gs[GR_B_TTL, :]),
            "Ваши персональные оценки студентами", "Б"
        )

        # Карточки раздела Б (4 карточки, каждая = 3 из 12 колонок)
        b_color = score_color(t_mean, scale=10)
        b_cards = [
            ("Средний балл\n(шкала 0–10)",
             f"{t_mean:.2f}" if not pd.isna(t_mean) else "—",
             scale_label(t_mean, scale=10), b_color),
            ("Медианный балл",
             f"{t_median:.1f}" if not pd.isna(t_median) else "—",
             "медиана оценок", GRAY),
            ("Количество\nответов",
             str(t_n), "студентов оценили вас", PRIMARY),
            ("Программ\n& курсов",
             str(len(program_rows)), "программ / курсов", PRIMARY),
        ]
        for i, (ttl, val, sub, clr) in enumerate(b_cards):
            c_start = i * 3
            ax = fig.add_subplot(gs[GR_B_0, c_start:c_start + 3])
            draw_metric_card(ax, ttl, val, sub, clr)

        # Заголовок таблицы программ
        draw_section_title(
            fig.add_subplot(gs[GR_TBL_T, :]),
            "Детализация по программам"
        )

        # Таблица
        draw_programs_table(fig.add_subplot(gs[GR_TBL, :]), program_rows)

        # Подвал
        fig.text(0.50, 0.010,
                 f"Конфиденциально · Только для внутреннего использования · "
                 f"Сгенерировано {date_str}",
                 ha="center", fontsize=6.5, color=GRAY)

        pdf.savefig(fig, bbox_inches="tight")
        plt.close(fig)

        # ================================================================
        # СТРАНИЦА 2: гистограмма оценок + статистика
        # ================================================================
        fig2 = plt.figure(figsize=(8.27, 11.69))
        fig2.subplots_adjust(left=0.10, right=0.90, top=0.93, bottom=0.10)
        gs2 = gridspec.GridSpec(3, 1, figure=fig2,
                                height_ratios=[0.12, 1.5, 0.55],
                                hspace=0.50)

        # Подзаголовок страницы
        ax2_hdr = fig2.add_subplot(gs2[0])
        ax2_hdr.axis("off")
        ax2_hdr.text(0.0, 0.75, teacher_name,
                     fontsize=14, fontweight="bold", color=PRIMARY)
        ax2_hdr.text(0.0, 0.10, "Подробная статистика оценок",
                     fontsize=9, color=TEXT_MID)

        # Гистограмма
        draw_rating_histogram(fig2.add_subplot(gs2[1]), ratings, t_mean)

        # Плитки с цифрами
        ax2_stats = fig2.add_subplot(gs2[2])
        ax2_stats.axis("off")
        ax2_stats.set_xlim(0, 1); ax2_stats.set_ylim(0, 1)
        q25, q75 = (ratings.quantile([0.25, 0.75])
                    if t_n > 0 else (np.nan, np.nan))

        stat_tiles = [
            ("Среднее",           f"{t_mean:.2f}"          if not pd.isna(t_mean)   else "—"),
            ("Медиана",           f"{t_median:.1f}"         if not pd.isna(t_median) else "—"),
            ("Q1 / Q3",           f"{q25:.1f} / {q75:.1f}" if t_n > 0               else "—"),
            ("Высокие (9–10)",    f"{perc_high:.0f}%"),
            ("Низкие (0–4)",      f"{perc_low:.0f}%"),
            ("Всего ответов",     str(t_n)),
        ]

        for i, (lbl, val) in enumerate(stat_tiles):
            col_i = i % 3
            row_i = i // 3
            x = 0.04 + col_i * 0.33
            y = 0.78  - row_i * 0.44
            ax2_stats.add_patch(mpatches.FancyBboxPatch(
                (x, y - 0.20), 0.28, 0.30,
                boxstyle="round,pad=0.01",
                facecolor=LIGHT_BG, edgecolor=GRAY, linewidth=0.8,
            ))
            ax2_stats.text(x + 0.14, y + 0.02, val,
                           ha="center", va="center",
                           fontsize=15, fontweight="bold",
                           color=PRIMARY if i < 3 else TEXT_MID)
            ax2_stats.text(x + 0.14, y - 0.12, lbl,
                           ha="center", va="center",
                           fontsize=7.5, color=TEXT_MID)

        fig2.text(0.50, 0.018,
                  f"Конфиденциально · Только для внутреннего использования · "
                  f"Сгенерировано {date_str}",
                  ha="center", fontsize=6.5, color=GRAY)

        pdf.savefig(fig2, bbox_inches="tight")
        plt.close(fig2)


print("Функция generate_pdf_for_teacher определена.")

In [ ]:
# ── Загрузка данных ───────────────────────────────────────────────────────
print("Загрузка данных...")

general_df  = pd.read_csv(GENERAL_CSV)
teachers_df = pd.read_csv(TEACHERS_CSV)
teachers_df["rating"] = pd.to_numeric(teachers_df["rating"], errors="coerce")

print(f"  Общий опрос:     {len(general_df):,} строк")
print(f"  Оценки учителей: {len(teachers_df):,} строк, "
      f"{teachers_df['teacher'].nunique()} преподавателей")

# Список преподавателей к генерации
counts = teachers_df.groupby("teacher")["rating"].count()
if ONLY_TEACHER:
    teachers_to_process = [ONLY_TEACHER]
else:
    teachers_to_process = sorted(counts[counts >= MIN_RESPONSES].index.tolist())

print(f"  Будет сгенерировано PDF: {len(teachers_to_process)} "
      f"(min {MIN_RESPONSES} ответов)")

In [ ]:
# ── Генерация PDF ─────────────────────────────────────────────────────────
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

success, errors = 0, 0
total = len(teachers_to_process)

for i, teacher_name in enumerate(teachers_to_process, 1):
    t_df  = teachers_df[teachers_df["teacher"] == teacher_name].copy()
    fname = safe_filename(teacher_name) + ".pdf"
    fpath = str(OUTPUT_DIR / fname)

    try:
        generate_pdf_for_teacher(teacher_name, t_df, general_df, fpath)
        success += 1
        if i % 20 == 0 or i == total:
            print(f"  [{i:>3}/{total}] {success} готово, {errors} ошибок")
    except Exception as e:
        errors += 1
        print(f"  [{i:>3}/{total}] ✗ {teacher_name}: {e}")

print(f"\nГотово: {success} PDF сгенерировано, {errors} ошибок.")
print(f"Папка: {OUTPUT_DIR.resolve()}")